# Class 1 — ChatGPT & Claude as Workflow Engines
**Week 3: No-Code & Low-Code AI Builders — "The Workflow Canvas"**

### Learning objectives
By the end of this notebook you will be able to:
- Treat every chat turn as a **trigger → action** loop and chain multiple turns on purpose
- Simulate Project instructions with a system prompt that persists across turns
- Build a small **prompt library** of reusable templates with placeholders
- Call Groq from Python (same roles as the chat UI) and compare two models side by side

Live cells need a `GROQ_API_KEY` (see Setup). Conceptual / writing cells work without one.

Run each cell in order with **Shift+Enter**.

## Setup
**Running in Google Colab:**
1. Get a free key from https://console.groq.com/keys
2. Click the key icon (🔑 Secrets) in the left sidebar
3. Add a secret named `GROQ_API_KEY`, paste your key, toggle **Notebook access** on
4. Run the two setup cells below

**Elsewhere:** set `GROQ_API_KEY` as an environment variable before launching Jupyter.

In [ ]:
!pip install -q groq

In [ ]:
import os

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    print(
        "No API key found — discussion and writing cells still work.\n"
        "In Colab: add a secret named GROQ_API_KEY via the 🔑 Secrets panel and enable notebook access.\n"
        "Elsewhere: set GROQ_API_KEY as an environment variable before launching Jupyter."
    )
else:
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY
    print("GROQ_API_KEY loaded — live cells will work.")

## 1. Chat Is Already a Workflow Engine
Every message you send is a **trigger**. Every reply is an **action** whose output becomes context for the next trigger. The chat UI hides the plumbing; the cell below makes it explicit: we keep a `messages` list and append each turn.

In [ ]:
def call_llm(messages, model="llama-3.3-70b-versatile", max_tokens=300, temperature=0.4):
    """Send a multi-turn message list to Groq and return assistant text, or an error string."""
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        return "Error: GROQ_API_KEY is not set."
    try:
        from groq import Groq
        client = Groq(api_key=api_key)
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            max_tokens=max_tokens,
            temperature=temperature,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error calling Groq: {e}"


# Same "trigger" you'd type into ChatGPT / Claude — sent via code
messages = [
    {"role": "system", "content": "You are a concise writing coach. Keep replies under 80 words."},
    {"role": "user", "content": "Outline a blog post about no-code AI for beginners."},
]

if os.environ.get("GROQ_API_KEY"):
    reply = call_llm(messages)
    print("--- assistant ---")
    print(reply)
else:
    print("Set GROQ_API_KEY, then re-run this cell to see the model's reply.")

## 2. A Multi-Turn Trigger → Action Chain
In the UI you'd send follow-ups in the same thread. In code you append the assistant reply, then add another user message. Below we run a three-turn blog workflow: outline → draft one section → rewrite for tone.

In [ ]:
workflow = [
    {"role": "system", "content": "You are a concise writing coach. Keep each reply under 100 words."},
]

turns = [
    "Outline a 4-section blog post about no-code AI for beginners.",
    "Using that outline, write only section 2 as a short draft paragraph.",
    "Rewrite section 2 in a warmer, more conversational tone. Keep it under 60 words.",
]

if os.environ.get("GROQ_API_KEY"):
    for i, user_text in enumerate(turns, start=1):
        workflow.append({"role": "user", "content": user_text})
        reply = call_llm(workflow, max_tokens=220)
        workflow.append({"role": "assistant", "content": reply})
        print(f"=== Turn {i}: trigger ===")
        print(user_text)
        print(f"--- action ---")
        print(reply)
        print()
else:
    print("Set GROQ_API_KEY, then re-run this cell to walk the 3-turn workflow.")

## 3. Project Instructions as a Persistent System Prompt
ChatGPT/Claude **Projects** store standing rules (tone, format, constraints) so you don't re-type them every chat. In the API, that maps to a system message that stays at the front of every conversation. Same user ask, two different "project" instruction sets:

In [ ]:
user_ask = "Summarize this meeting note into action items:\n\nWe agreed Maya owns the landing page copy by Friday. Sam will check analytics. Next sync Tuesday."

project_a = (
    "You are an executive assistant for a product team. "
    "Always reply as a numbered list of action items. "
    "Each item: owner, task, deadline if mentioned. No intro, no closing."
)
project_b = (
    "You are a playful standup bot for a startup Slack channel. "
    "Summarize in exactly 2 short bullet points with a light emoji. "
    "Keep the whole reply under 40 words."
)

if os.environ.get("GROQ_API_KEY"):
    for label, system in [("Project A — exec assistant", project_a), ("Project B — Slack bot", project_b)]:
        msgs = [
            {"role": "system", "content": system},
            {"role": "user", "content": user_ask},
        ]
        print(f"=== {label} ===")
        print(call_llm(msgs, max_tokens=180))
        print()
else:
    print("Set GROQ_API_KEY, then re-run this cell to compare the two project instruction sets.")

## 4. Prompt Libraries: Templates with Placeholders
A **prompt library** is a saved set of reusable templates so you stop reinventing the same instructions. Below is a tiny library: fill placeholders, then send.

In [ ]:
PROMPT_LIBRARY = {
    "meeting_actions": (
        "Summarize the following transcript into exactly 3 action items for {audience}.\n"
        "Tone: {tone}. Format: numbered list, each item starts with a verb.\n\n"
        "TRANSCRIPT:\n{transcript}"
    ),
    "rewrite_tone": (
        "Rewrite the text below for a {audience} in a {tone} tone.\n"
        "Keep roughly the same length. Do not add new facts.\n\n"
        "TEXT:\n{text}"
    ),
}


def fill_template(name, **kwargs):
    """Look up a library template and fill its placeholders."""
    if name not in PROMPT_LIBRARY:
        raise KeyError(f"Unknown template: {name}. Available: {list(PROMPT_LIBRARY)}")
    return PROMPT_LIBRARY[name].format(**kwargs)


filled = fill_template(
    "meeting_actions",
    audience="engineering managers",
    tone="crisp and direct",
    transcript="Blocked on API keys. Priya will request Groq access today. Demo slides due Wednesday.",
)
print("--- filled template ---")
print(filled)

if os.environ.get("GROQ_API_KEY"):
    print("\n--- model reply ---")
    print(call_llm([{"role": "user", "content": filled}], max_tokens=180))
else:
    print("\nSet GROQ_API_KEY, then re-run this cell to send the filled template to the model.")

## 5. Compare Two Engines (via Groq)
In the chat products, ChatGPT and Claude feel different — tone, structure, caution. Here we approximate that idea by sending the **same** prompt to two Groq-hosted models and reading the differences. Neither is "better"; pick based on the task.

In [ ]:
compare_prompt = (
    "A teammate asked: 'Should we put every client conversation in one giant Project chat?' "
    "Answer in 3 short bullets. Be practical."
)

models = [
    ("llama-3.3-70b-versatile", "larger / more deliberate"),
    ("llama-3.1-8b-instant", "smaller / snappier"),
]

if os.environ.get("GROQ_API_KEY"):
    for model, note in models:
        msgs = [
            {"role": "system", "content": "You are a clear AI workflow coach."},
            {"role": "user", "content": compare_prompt},
        ]
        print(f"=== {model} ({note}) ===")
        print(call_llm(msgs, model=model, max_tokens=200))
        print()
else:
    print("Set GROQ_API_KEY, then re-run this cell to compare the two models.")

### Week 3, Class 1 — closed
You can now see chat as a loop, simulate Project instructions in code, reuse prompts from a library, and call Groq the same way the chat window does under the hood. Class 2 packages those instructions into a reusable Custom GPT / Assistant.

## Challenges
Work through these in order. No solutions are provided — each starter cell has a `# TODO` marking where your code goes. Challenges 1–3 need `GROQ_API_KEY`.

### Challenge 1 — Build a 3-Turn Workflow Chain
Pick a real task (e.g. job application email, product FAQ, study plan). Drive it through **three** user turns where each turn depends on the previous assistant reply. Print each trigger and action.

**Acceptance criteria:** prints Turn 1 / 2 / 3 with both the user trigger and the model action for each.

In [ ]:
# TODO: build a messages list with a system prompt, append 3 user turns,
# call call_llm after each, append assistant replies, and print every turn

### Challenge 2 — Design a Prompt-Library Template
Add a new template to `PROMPT_LIBRARY` (or a new dict) with at least **two** placeholders. Fill it for a concrete use case, print the filled prompt, and send it via `call_llm`.

**Acceptance criteria:** prints the filled template and the model reply; template name is not one of the two examples from Section 4.

In [ ]:
# TODO: define a new template with placeholders, fill it, call call_llm, print both

### Challenge 3 — Compare Two Groq Models
Send the **same** user prompt (your choice) to `llama-3.3-70b-versatile` and `llama-3.1-8b-instant`. Print both answers and one sentence on how they differ (tone, length, structure, or caution).

**Acceptance criteria:** prints both model names + replies, plus a one-line comparison note you wrote.

In [ ]:
# TODO: call call_llm twice with the same messages but different model= values, then print your note

### Challenge 4 — Write Project Instructions for a Real Use Case
Without calling the API: write Project-style standing instructions (8–12 lines) for one real use case — e.g. "Client X brand voice", "office-hours tutor", or "bug-triage helper". Include tone, format, what to always do, and what to never do.

*(Write your Project instructions here.)*